# Sales & Demand Forecasting System
## Feature Engineering

**Author:** Project Maintainer
**Program:** Future Interns Machine Learning Internship 2026  
**CIN ID:** N/A
**Phase:** 2 of 3 — Feature Engineering  

---

## Objectives

1. Aggregate raw transaction data into monthly sales figures
2. Create time based features for the forecasting model
3. Engineer lag features capturing previous months sales
4. Create rolling average features capturing sales trends
5. Prepare and save the final modelling dataset

---

## Why Feature Engineering Matters for Forecasting

Raw transaction data cannot be fed directly into a 
forecasting model. We must:

- Aggregate daily transactions into monthly totals
- Create features that capture time patterns
- Add lag values so the model learns from history
- Add rolling averages so the model learns trends

These engineered features are what allow the model
to make accurate future predictions.

In [1]:
# ─── Imports ──────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported sucessfully")

Libraries imported sucessfully


In [2]:
# ─── Imports ──────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ─── Load Dataset ─────────────────────────────────────────────
df = pd.read_csv(
    '../data/raw/Sample - Superstore.csv',
    encoding='latin-1'
)

# ─── Convert dates ────────────────────────────────────────────
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date']  = pd.to_datetime(df['Ship Date'])

print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")

Dataset loaded successfully!
Shape: (9994, 21)


## 1. Monthly Sales Aggregation

We aggregate all individual transactions into
monthly total sales figures.

This transforms 9,994 transaction rows into
48 monthly data points — one per month from
January 2014 to December 2017.

In [3]:
# ─── Aggregate to monthly sales ───────────────────────────────
df['Year']       = df['Order Date'].dt.year
df['Month']      = df['Order Date'].dt.month
df['Month_Year'] = df['Order Date'].dt.to_period('M')

monthly_df = df.groupby('Month_Year').agg(
    Total_Sales   = ('Sales', 'sum'),
    Total_Profit  = ('Profit', 'sum'),
    Total_Orders  = ('Order ID', 'count'),
    Total_Quantity= ('Quantity', 'sum')
).reset_index()

monthly_df['Month_Year'] = monthly_df['Month_Year'].astype(str)
monthly_df['Month_Year'] = pd.to_datetime(monthly_df['Month_Year'])
monthly_df = monthly_df.sort_values('Month_Year').reset_index(drop=True)

monthly_df['Year']    = monthly_df['Month_Year'].dt.year
monthly_df['Month']   = monthly_df['Month_Year'].dt.month
monthly_df['Quarter'] = monthly_df['Month_Year'].dt.quarter

print("Monthly aggregation complete!")
print(f"Shape: {monthly_df.shape}")
print(f"\nFirst 5 rows:")
print(monthly_df.head())

Monthly aggregation complete!
Shape: (48, 8)

First 5 rows:
  Month_Year  Total_Sales  Total_Profit  Total_Orders  Total_Quantity  Year  \
0 2014-01-01    14236.895     2450.1907            79             284  2014   
1 2014-02-01     4519.892      862.3084            46             159  2014   
2 2014-03-01    55691.009      498.7299           157             585  2014   
3 2014-04-01    28295.345     3488.8352           135             536  2014   
4 2014-05-01    23648.287     2738.7096           122             466  2014   

   Month  Quarter  
0      1        1  
1      2        1  
2      3        1  
3      4        2  
4      5        2  


In [4]:
df.columns

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit', 'Year',
       'Month', 'Month_Year'],
      dtype='object')

In [5]:
monthly_df.head(5)

,Month_Year,Total_Sales,Total_Profit,Total_Orders,Total_Quantity,Year,Month,Quarter
0,2014-01-01,14236.895,2450.1907,79,284,2014,1,1
1,2014-02-01,4519.892,862.3084,46,159,2014,2,1
2,2014-03-01,55691.009,498.7299,157,585,2014,3,1
3,2014-04-01,28295.345,3488.8352,135,536,2014,4,2
4,2014-05-01,23648.287,2738.7096,122,466,2014,5,2


## 2. Lag Features

Lag features capture historical sales values
as input variables for the model.

Lag 1 = sales from 1 month ago
Lag 2 = sales from 2 months ago
Lag 3 = sales from 3 months ago

This allows the model to learn:
"If sales were high last month,
what will they be this month?"

In [6]:
# ─── Create lag features ──────────────────────────────────────
monthly_df['Lag_1'] = monthly_df['Total_Sales'].shift(1)
monthly_df['Lag_2'] = monthly_df['Total_Sales'].shift(2)
monthly_df['Lag_3'] = monthly_df['Total_Sales'].shift(3)

# ─── Rolling average features ─────────────────────────────────
monthly_df['Rolling_3']  = monthly_df['Total_Sales'].shift(1).rolling(window=3).mean()
monthly_df['Rolling_6']  = monthly_df['Total_Sales'].shift(1).rolling(window=6).mean()

# ─── Month over month growth ──────────────────────────────────
monthly_df['MoM_Growth'] = monthly_df['Total_Sales'].pct_change() * 100

print("Lag and rolling features created!")
print(f"\nNew shape: {monthly_df.shape}")
print(f"\nFirst 8 rows:")
print(monthly_df[['Month_Year', 'Total_Sales', 
                   'Lag_1', 'Lag_2', 'Lag_3',
                   'Rolling_3', 'Rolling_6',
                   'MoM_Growth']].head(8))

Lag and rolling features created!

New shape: (48, 14)

First 8 rows:
  Month_Year  Total_Sales       Lag_1       Lag_2      Lag_3     Rolling_3  \
0 2014-01-01   14236.8950         NaN         NaN        NaN           NaN   
1 2014-02-01    4519.8920  14236.8950         NaN        NaN           NaN   
2 2014-03-01   55691.0090   4519.8920  14236.8950        NaN           NaN   
3 2014-04-01   28295.3450  55691.0090   4519.8920  14236.895  24815.932000   
4 2014-05-01   23648.2870  28295.3450  55691.0090   4519.892  29502.082000   
5 2014-06-01   34595.1276  23648.2870  28295.3450  55691.009  35878.213667   
6 2014-07-01   33946.3930  34595.1276  23648.2870  28295.345  28846.253200   
7 2014-08-01   27909.4685  33946.3930  34595.1276  23648.287  30729.935867   

      Rolling_6   MoM_Growth  
0           NaN          NaN  
1           NaN   -68.252263  
2           NaN  1132.131409  
3           NaN   -49.192257  
4           NaN   -16.423401  
5           NaN    46.290205  
6  26831.0

## 3. Drop NaN rows and prepare final dataset

The first few rows contain NaN values
because lag and rolling features require
historical data that does not exist
for the earliest months.

We drop these rows to ensure the model
trains only on complete data.

In [7]:
# ─── Drop NaN rows ────────────────────────────────────────────
df_model = monthly_df.dropna().reset_index(drop=True)

print(f"Rows before dropping NaN: {len(monthly_df)}")
print(f"Rows after dropping NaN:  {len(df_model)}")
print(f"\nFinal dataset shape: {df_model.shape}")
print(f"\nDate range:")
print(f"From: {df_model['Month_Year'].min().date()}")
print(f"To:   {df_model['Month_Year'].max().date()}")
print(f"\nColumns in final dataset:")
for col in df_model.columns:
    print(f"  → {col}")

Rows before dropping NaN: 48
Rows after dropping NaN:  42

Final dataset shape: (42, 14)

Date range:
From: 2014-07-01
To:   2017-12-01

Columns in final dataset:
  → Month_Year
  → Total_Sales
  → Total_Profit
  → Total_Orders
  → Total_Quantity
  → Year
  → Month
  → Quarter
  → Lag_1
  → Lag_2
  → Lag_3
  → Rolling_3
  → Rolling_6
  → MoM_Growth


## 4. Save Processed Dataset

We save the engineered dataset to the
processed folder for use in model training.

In [8]:
# ─── Save processed dataset ───────────────────────────────────
df_model.to_csv('../data/processed/monthly_sales_features.csv', 
                index=False)

print("Processed dataset saved successfully!")
print(f"File: data/processed/monthly_sales_features.csv")
print(f"Shape: {df_model.shape}")
print(f"\nFinal feature summary:")
print(df_model.describe().round(2))

Processed dataset saved successfully!
File: data/processed/monthly_sales_features.csv
Shape: (42, 14)

Final feature summary:
                          Month_Year  Total_Sales  Total_Profit  Total_Orders  \
count                             42        42.00         42.00         42.00   
mean   2016-03-16 18:51:25.714285824     50862.25       6461.47        221.90   
min              2014-07-01 00:00:00     11951.41      -3281.01         58.00   
25%              2015-05-08 18:00:00     31417.04       3363.73        153.50   
50%              2016-03-16 12:00:00     44116.24       5849.20        200.00   
75%              2017-01-24 06:00:00     72443.92       9026.61        289.25   
max              2017-12-01 00:00:00    118447.82      17885.31        462.00   
std                              NaN     24813.17       4328.94        104.04   

       Total_Quantity     Year  Month  Quarter      Lag_1     Lag_2     Lag_3  \
count           42.00    42.00  42.00    42.00      42.00     4

## Feature Engineering Summary

The following features were engineered for the
forecasting model:

**Time Features:**
→ Year, Month, Quarter — capture time patterns

**Lag Features:**
→ Lag_1 — sales from 1 month ago
→ Lag_2 — sales from 2 months ago
→ Lag_3 — sales from 3 months ago

**Rolling Average Features:**
→ Rolling_3 — 3 month moving average
→ Rolling_6 — 6 month moving average

**Growth Features:**
→ MoM_Growth — month over month percentage change

**Final Dataset:**
→ 42 complete months of data
→ July 2014 to December 2017
→ 14 features ready for modelling
→ Saved to data/processed/monthly_sales_features.csv

Feature Engineering Complete — Proceeding to Model Training